# 03 Preprocessing

This notebook prepares the cleaned CredData lines for modeling in Phase 2. It reads `data/processed/credata_clean.csv` from `01_cleaning`, turns each row into numbers with `src/features.py`, splits the rows by repository, applies a log transform to the skewed length and position measures, and scales the numeric features with a scaler fitted on the train set only.

No model is trained here. Outputs show counts and summary numbers only, never `LineText` or `Value`, because the notebook is tracked by git.

## Setup

We import the libraries and add the `src` folder to the path so the notebook can use `load_data` and `build_features` from `src/features.py`. The notebook runs from the `notebooks` folder, so paths go up one level.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# The notebook runs from the notebooks folder so src and data sit one level up
sys.path.insert(0, str(Path("..") / "src"))
from features import build_features, load_data

CLEAN_PATH = Path("..") / "data" / "processed" / "credata_clean.csv"
OUT_DIR = Path("..") / "data" / "processed"

# Colors for the train and test sets in every chart
TRAIN_COLOR = "#2a78d6"
TEST_COLOR = "#eb6834"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## Load the cleaned data and build features

`load_data` reads the cleaned file with `dtype=str` and `keep_default_na=False`, as agreed in `docs/CLEANED_DATA.md`, and turns `label` into a whole number. `build_features` works out every feature from its own row, so it can run before the split without letting test rows affect train rows.

In [2]:
df = load_data(CLEAN_PATH)
X = build_features(df)
print("rows:", len(df))
print("feature columns:", X.shape[1])
print("missing feature values:", int(X.isna().sum().sum()))
print("label 1 share:", round(df["label"].mean() * 100, 2), "percent")

rows: 66291
feature columns: 39
missing feature values: 0
label 1 share: 23.34 percent


All 66,291 cleaned rows give 39 features with no missing values, and 23.34 percent of rows are real secrets, the same share as in `01_cleaning`.

## Feature types

A feature that only takes the values 0 and 1 is a flag and is already on a common scale. The other features are counts, positions, ratios or entropy, and these are the ones that need a transform or scaling.

In [3]:
binary_cols = [c for c in X.columns if set(X[c].unique()) <= {0, 1}]
numeric_cols = [c for c in X.columns if c not in binary_cols]
print("flag features:", len(binary_cols))
print("numeric features:", len(numeric_cols))
print()
print(X[numeric_cols].describe().T[["min", "25%", "50%", "75%", "max"]].round(3).to_string())

flag features: 28
numeric features: 11

                    min   25%    50%       75%         max
value_len           0.0   0.0    0.0    30.000    6368.000
value_entropy       0.0   0.0    0.0     3.625       6.022
value_digit_ratio   0.0   0.0    0.0     0.162       1.000
value_upper_ratio   0.0   0.0    0.0     0.000       1.000
value_lower_ratio   0.0   0.0    0.0     0.417       1.000
value_symbol_ratio  0.0   0.0    0.0     0.062       1.000
line_len            9.0  38.0   58.0    95.000  711242.000
line_number         1.0  66.0  255.0  1688.000  917380.000
lines_spanned       1.0   1.0    1.0     1.000     106.000
var_name_len        0.0   3.0    8.0    13.000     214.000
path_depth          3.0   3.0    3.0     4.000      10.000


28 of the 39 features are flags. Most value measures have a median of 0 because about half of the rows have no value, where the whole line is marked. `line_len`, `line_number` and `value_len` have maximums thousands of times their medians, so they need a transform before scaling.